In [0]:
%run ./00_config

In [0]:
import os, sys
from datetime import date

# Permite importar src/ desde la raíz del repositorio
raiz_repo = os.path.abspath("..")
if raiz_repo not in sys.path:
    sys.path.append(raiz_repo)

from src.openmeteo_api import descargar_periodo

dbutils.widgets.text("fecha_desde", "2022-01-01", "Fecha desde")
dbutils.widgets.text("fecha_hasta", "2022-12-31", "Fecha hasta")
FECHA_DESDE = date.fromisoformat(dbutils.widgets.get("fecha_desde"))
FECHA_HASTA = date.fromisoformat(dbutils.widgets.get("fecha_hasta"))

In [0]:
from src.openmeteo_api import LimiteApiAlcanzado, descargar_periodo

try:
    resumen = descargar_periodo(
        desde=FECHA_DESDE,
        hasta=FECHA_HASTA,
        carpeta_destino=RUTAS["openmeteo"],
        carpeta_fallback=os.path.join(raiz_repo, "muestras", "openmeteo"),
    )
except LimiteApiAlcanzado as error:
    raise RuntimeError(f"\n{error}") from None      # detiene la celda con el mensaje

display(spark.createDataFrame(list(resumen.items()), ["fichero", "origen"]))
assert "error" not in resumen.values(), "Hay meses sin datos ni en la API ni en el fallback"

In [0]:
from pyspark.sql import functions as F

bronze_openmeteo = (
    spark.read
    .option("multiLine", True)              # cada fichero es un único JSON de varias líneas
    .option("primitivesAsString", True)     # todo como texto, tal cual llega
    .json(f"{RUTAS['openmeteo']}/*.json")
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(bronze_openmeteo.write
    .mode("overwrite")                      # idempotente: relanzar no duplica filas
    .option("overwriteSchema", True)
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.bronze_openmeteo"))

display(spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_openmeteo")
        .select("_source_file", "_ingest_ts", "latitude", "longitude"))